In [1]:
import pandas as pd

df = pd.read_csv("data/raw/kredit_sumatera.csv")

# Pertanyaan 1: seberapa tidak seimbang targetnya?
print("Tingkat gagal bayar:", f"{df.gagal_bayar.mean():.2%}")

# Pertanyaan 2: apakah populasi berubah sepanjang waktu?
print("\nProporsi segmen per tahun:")
print(pd.crosstab(df.periode.str[:4], df.segmen, normalize="index").round(3))

# Pertanyaan 3: kolom mana yang punya nilai hilang, dan mengapa?
print("\nKolom dengan nilai hilang:")
hilang = df.isna().mean().sort_values(ascending=False)
print(hilang[hilang > 0].round(4))
print("\nProporsi nilai hilang rata_saldo_90h per segmen:")
print(df.groupby("segmen")["rata_saldo_90h"].apply(lambda s: s.isna().mean()).round(3))

# Pertanyaan 4: adakah kolom yang mencurigakan?
print("\nProporsi terisi status_restrukturisasi berdasarkan gagal_bayar:")
print(df.groupby("gagal_bayar")["status_restrukturisasi"].apply(lambda s: s.notna().mean()))

Tingkat gagal bayar: 8.30%

Proporsi segmen per tahun:
segmen   nelayan  pedagang_musiman  pegawai_tetap  petani  wiraswasta
periode                                                              
2022       0.067             0.139          0.421    0.13       0.242
2023       0.067             0.142          0.419    0.13       0.241
2024       0.099             0.171          0.339    0.16       0.231

Kolom dengan nilai hilang:
status_restrukturisasi    0.9479
lama_usaha_tahun          0.0294
rata_saldo_90h            0.0231
dtype: float64

Proporsi nilai hilang rata_saldo_90h per segmen:
segmen
nelayan             0.108
pedagang_musiman    0.000
pegawai_tetap       0.000
petani              0.105
wiraswasta          0.000
Name: rata_saldo_90h, dtype: float64

Proporsi terisi status_restrukturisasi berdasarkan gagal_bayar:
gagal_bayar
0    0.000000
1    0.627509
Name: status_restrukturisasi, dtype: float64


In [2]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Membuat fitur rasio angsuran dan menyiapkan matriks fitur X serta target y
df["rasio_angsuran"] = df.angsuran_bulanan / df.penghasilan_bulanan
num = [
    "usia", "tanggungan", "lama_usaha_tahun", "penghasilan_bulanan",
    "ragam_penghasilan", "plafon_diajukan", "tenor_bulan", "bunga_tahunan",
    "rasio_angsuran", "rata_saldo_90h", "jumlah_tunggakan_12b"
]
X = df[num].join(pd.get_dummies(df[["segmen", "wilayah", "kanal"]]))
y = df.gagal_bayar

# Langkah 5: Pembagian MENURUT WAKTU (out-of-time)
latih = df.periode <= "2023-12"
uji = df.periode > "2024-06"

m = HistGradientBoostingClassifier(
    max_depth=6, learning_rate=0.06, max_iter=250, random_state=42
)
m.fit(X[latih], y[latih])
skor = m.predict_proba(X[uji])[:, 1]
print("AUC out-of-time:", round(roc_auc_score(y[uji], skor), 4))

# Langkah 6: Perbandingan dengan pembagian acak 80:20
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
m2 = HistGradientBoostingClassifier(
    max_depth=6, learning_rate=0.06, max_iter=250, random_state=42
).fit(Xa, ya)
print("AUC pembagian acak:", round(roc_auc_score(yb, m2.predict_proba(Xb)[:, 1]), 4))

AUC out-of-time: 0.7744
AUC pembagian acak: 0.7631
